# Lesson 4, Part 2

initial imports and helper functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def eval_poly(x, coef):
    poly = coef[-1]
    for c in coef[-2::-1]:
        poly = poly * x + c
    return poly

def coef_poly(coef, power=None):
    if power == None:
        return np.array(coef)
    n = max(power)
    coef_full = np.zeros(n + 1)
    coef_full[power] = coef
    return coef_full

def deriv_poly(coef):
    if len(coef) == 1: return [0]
    return coef[1:] * np.arange(1, len(coef))

Let's start by loading some functions of polynomials from the previous lesson. 
They are refactored slightly to compress. The integral uses **np.concatenate** to join arrays into larger arrays.

In [ ]:
def deriv_poly(coef,order=1):
    for i in range(order):
        if len(coef)==1:
            return [0]
        coef = coef[1:] * np.arange(1,len(coef))
    return coef

def integral_poly(coef,order=1):      # ignores integration constant
    for i in range(order):
        coef = coef / np.arange(1,len(coef)+1)
        coef = np.concatenate(([0], coef)) # form larger array with 0 followed by coef
    return coef   

# demo
coef = coef_poly([1],[4])
x = np.linspace(1,3,3)
print('x =',x)
for coef in coef_poly([1],[4]),coef_poly([2,6,12]):
    print('\ncoef =',coef,', values',eval_poly(x,coef));
    cd1 = deriv_poly(coef) 
    print('coef 1st derivative',cd1,', values',eval_poly(x,cd1))
    cd2 = deriv_poly(cd1)
    print('coef 2nd derivative',cd2,', values',eval_poly(x,cd2))

The codes below extend this to cover any sum of power functions. 
Here is a helper function that eliminates any coefficient of 0 along with the corresponding power.

In [ ]:
def sum_power_fn(x,coef,power):
    y = np.zeros(len(x))
    for i in range(len(coef)):
        y += coef[i] * x**power[i]
    return y

def deriv_sum_power_orig(coef,power,order=1):
    for i in range(order):
        coef = coef * power
        power = power - 1
    return coef,power

def integral_sum_power_orig(coef,power,order=1):  # ignores integration constant
    for i in range(order):
        power = power + 1
        coef = coef / power
    return coef,power
    
coef,power = np.array([2,2,3,3,4]),np.array([-2,-1,0,0.5,1])
print('coef =',coef,', power =',power)
cd,pd = deriv_sum_power_orig(coef,power)
print('coef_deriv =',cd,', power_deriv =',pd)
ci,pi = integral_sum_power_orig(coef,power)
print('coef_integral =',ci,', power_integral =',pi)

There are two problems with the preceding code. 
First, the derivative of a constant will be reported as 0 times *x*<sup>-1</sup>, which is undefined at *x* = 0. 
Second, the integral of 1/*x* generates a coefficient of infinity on a power 0. The code here corrects the first problem.

In [ ]:
def clean_coef_power_orig(coef,power):
    coef,power = np.array(coef),np.array(power)
    k = len(coef)
    while k > 0:
        k -= 1     # scroll backwards to keep the not-yet-examined indices from changing
        if coef[k] == 0:
            coef = np.delete(coef,k)    # deletes index k of coef and renames the remainder as coef
            power = np.delete(power,k)  # deletes index k of power and renames the remainder as power
    return coef,power
    
def clean_coef_power(coef,power):       # crisper version using masking
    coef,power = np.array(coef),np.array(power)
    indices_to_keep = (coef!=0)         # true except where coef==0
    power = power[indices_to_keep]      # mask eliminates indices where coef==0
    coef = coef[indices_to_keep]
    return coef,power 

def deriv_sum_power_fn(coef,power,order=1):
    for i in range(order):
        coef = coef * power
        power = power - 1
    return clean_coef_power(coef,power)
    
coef,power = np.array([2,2,3,3,4]),np.array([-2,-1,0,0.5,1])
cd,pd = deriv_sum_power_orig(coef,power)
cd0,pd0 = clean_coef_power_orig(cd,pd)
print('coef =',cd0,', power =',pd0)
cd1,pd1 = deriv_sum_power_fn(cd,pd)
print('coef_deriv =',cd1,', power_deriv =',pd1)

While we haven't yet figured out how to integrate 1/*x*, 
here are two ways to flag the problem better.

In [ ]:
def integral_sum_power_fn(coef,power,order=1):
    for i in range(order):
        power = power + 1
        if np.any(power == 0):   # true if and only if at least one index with power_integral==0 
            return "Can't yet solve this"
        coef = coef / power
    return coef,power

def integral_sum_power_alt(coef,power,order=1):
    for i in range(order):
        power = power + 1
        indices_to_keep = (power!=0)
        if np.all(indices_to_keep):     # true if and only if no indices with power==0 
            coef = coef / power
        else:
            print("Excluding integral of power -1 which we can't yet compute")
            power = power[indices_to_keep]
            coef = coef[indices_to_keep] / power
    return coef,power

coef,power = [1,10,3,2,0],[-0.5,-2,-1,0,1]
coef,power = clean_coef_power(coef,power)
print(integral_sum_power_fn(coef,power),'\n')
print(integral_sum_power_alt(coef,power))

Let's chart some functions alongside their first and second derivatives.

In [ ]:
def power_fn_and_derivs(xmin,xmax,coef,power):
    x = np.linspace(xmin,xmax,500)
    y = sum_power_fn(x,coef,power) 
    cd,pd = deriv_sum_power_fn(coef,power) 
    dydx = sum_power_fn(x,cd,pd) 
    cd2,pd2 = deriv_sum_power_fn(cd,pd)
    d2ydx2 = sum_power_fn(x,cd2,pd2) 
    return x,y,dydx,d2ydx2

def plot_fn_derivs(x,y,dydx,d2ydx2):
    plt.subplot(3,1,1)      # place 3 charts vertically starting from top 
    plt.title('Function and its first two derivatives')
    plt.plot(x,y)
    plt.ylabel('function')
    plt.subplot(3,1,2)      # place next chart in second position
    plt.plot(x,dydx)
    plt.ylabel('1st deriv')
    plt.subplot(3,1,3)      # place next chart in third position
    plt.plot(x,d2ydx2)
    plt.ylabel('2nd deriv')
    plt.show()
    
coef,power = clean_coef_power([20,-8,2,-0.1],[-0.5,1,2,3])
xmin,xmax = 1,15
x,y,dydx,d2ydx2 = power_fn_and_derivs(xmin,xmax,coef,power)
plot_fn_derivs(x,y,dydx,d2ydx2)

This code highlights the connection between the shape of a function and the sign of its slope.

In [ ]:
  
coef,power = clean_coef_power([20,-8,2,-0.1],[-0.5,1,2,3])
xmin,xmax = 1,15
x,y,dydx,d2ydx2 = power_fn_and_derivs(xmin,xmax,coef,power)
# chart sign of first derivative in background
plt.title('Function and impact of first derivative')
rising = (dydx>0) * np.max(y)   # = max(y) if and only if rising
plt.fill_between(x,rising,0,label='rising',color='pink')
plt.plot(x,y)
plt.legend()
plt.show()

This code highlights the connection between function shape and convexity.

In [ ]:
  
coef,power = clean_coef_power([20,-8,2,-0.1],[-0.5,1,2,3])
xmin,xmax = 1,15
x,y,dydx,d2ydx2 = power_fn_and_derivs(xmin,xmax,coef,power)
# chart sign of second derivative in background
plt.title('Function and impact of second derivative')
convex = (d2ydx2>0) * np.max(y)   # = max(y) if and only if convex
plt.fill_between(x,convex,0,label='convex',color='lavender')
plt.plot(x,y)   
plt.legend()
plt.show()

The code here combines both procedures. 
Feel free to change the parameters and impose other criteria for background coloring.

In [ ]:
def plot_fn_impact(x,y,z,z_label,z_color):
    positive_z = (z>0) * np.max(y)
    plt.fill_between(x,positive_z,0,label=z_label,color=z_color)
    plt.plot(x,y)
    plt.legend()
    
def plot_power_fn_shape(xmin,xmax,coef,power):
    x,y,dydx,d2ydx2 = power_fn_and_derivs(xmin,xmax,coef,power)
    plt.title('Function and Impact of First Two Derivatives')
    plt.subplot(2,1,1)
    plot_fn_impact(x,y,dydx,'rising','pink')
    plt.subplot(2,1,2)
    plot_fn_impact(x,y,d2ydx2,'convex','lavender')
    plt.show()
               
coef,power = clean_coef_power([20,-8,2,-0.1],[-0.5,1,2,3])
xmin,xmax = 1,15
plot_power_fn_shape(xmin,xmax,coef,power)

Changing topics, consider an arrow shot at initial speed *V* meters per second with initial slope *s* relative to level ground, 
Ignoring friction, horizontal speed remains *Vt*/sqrt(1+*s*<sup>2</sup>). 
Vertical speed is originally *s* times horizontal speed but decreases by *g*=9.81 meters per second every second until the arrow hits ground.
Let's track the trajectory for various *s*.

In [ ]:
def trajectory(init_speed,init_slope):
    gravity = 9.8067
    max_time_aloft = 2 * init_speed / gravity  # achieved when arrow shot straight up
    time = np.linspace(0,max_time_aloft,1000)
    # compute x and y assuming the arrow keeps flying
    init_speed_x = init_speed / np.sqrt(1 + init_slope**2)  # initial horizontal speed  
    init_speed_y = init_speed_x * init_slope                # initial vertical speed
    x = init_speed_x * time          # absent friction, horizontal speed never changes
    y = -0.5 * gravity * time * time + init_speed_y * time
    # adjust x and y to stop changing once arrow hits ground
    time_of_landing = 2 * init_speed_y / gravity  # solve y=0 for time>0
    final_x = init_speed_x * time_of_landing  
    x[y<0] = final_x   # wherever y<0, replace corresponding x with final_x
    y[y<0] = 0         # wherever y<0, replace y with 0                  
    return time,x,y
    
init_speed = 70
init_slopes = [1/8,1/4,1/3,1/2,2/3,1,3/2,2,3,4,8]
plt.title("Flight of an arrow")
plt.xlabel('distance in meters')
plt.ylabel('height in meters')
for slope in init_slopes:
    t,x,y = trajectory(init_speed,slope)
    plt.plot(x,y,label=str(round(slope,2)))
plt.legend()
plt.show()

If we'd like to save the different trajectories as ndarrays, 
the simplest method forms a list of lists and converts it. 
There are also numerous methods for stacking ndarrays, with two introduced below.

In [ ]:
init_speed = 70
init_slopes = [1/8,1/4,1/3,1/2,2/3,1,3/2,2,3,4,8]

# Method 1
x_list,y_list = [],[]    # x_list=y_list=[] would make lists permanently match
for slope in init_slopes:
    t,x,y = trajectory(init_speed,slope)
    x_list.append(x)
    y_list.append(y)
x2D_1 = np.array(x_list)
y2D_1 = np.array(y_list)
    
# Method 2
x2D_2,y2D_2 = [ ],[ ]
for slope in init_slopes:
    t,x,y = trajectory(init_speed,slope)
    if len(x2D_2)==0:
        x2D_2 = x
        y2D_2 = y
    else:
        x2D_2 = np.vstack((x2D_2,x))   # need double parentheses
        y2D_2 = np.vstack((y2D_2,y))      
        
# Method 3
nslopes = len(init_slopes)
x2D_3 = np.zeros((nslopes,len(x)))        # need layers of parentheses 
y2D_3 = np.zeros((nslopes,len(x))) 
for i in range(nslopes):
    t,x,y = trajectory(init_speed,init_slopes[i])
    x2D_3[i] = x
    y2D_3[i] = y

print('x_list has length',len(x_list),'and x_list[i] has length',len(x_list[1]))
print('\nx2D_1 has shape',x2D_1.shape,', which means it has',x2D_1.shape[0],'rows and',x2D_1.shape[1],'columns')
print('\nDo x2D_1, x2D_2 and x2D_3 match?',np.all(x2D_1==x2D_2) and np.all(x2D_2==x2D_3))

One advantage of ndarrays over lists of lists is simpler indexing. 
 We can access element [i][j] by typing [i,j] and 
 we can access column j by typing [:j]. 
 We can also slice rows and columns individually.

In [ ]:
arr = np.array([[0,1,2,3],[10,11,12,13],[20,21,22,23]])
print('arr has shape',arr.shape,'with elements\n',arr)
print('\narr[0,1] =',arr[0,1],', arr[2,3]=',arr[2,3],', arr[-1,-1]=',arr[-1,-1])
print('\narr[:,2] =',arr[:,2])
print('\narr[:2,1:] =\n',arr[:2,1:])

Creative use of ndarrays can simplify many computations and numpy offers many functions that can help. 
One of those functions is **np.outer**, which forms a 2D multiplication table out of 1D arrays. 
Two others are **np.maximum** and **np.minimum**, which compare two equal-sized arrays element by element or compare one array to a constant.

In [ ]:
a = np.array([1,2,3,4])
b = np.array([1,2,3,4,5])
c = np.array([5,4,3,2,1])
print('a =',a,', b =',b,', c=',c)
print('\nnp.outer(a,b) =\n', np.outer(a,b))
print('\nnp.maximum(b,c) =', np.maximum(b,c))
print('\nnp.minimum(c,2.5) =', np.minimum(c,2.5))

We can use these functions to compute trajectories as 2D ndarrays 
without forming 1D ndarrays first. 
The main challenge is line up the dimensions correctly, 
without mixing up 1D arrays with 2D arrays or rows with columns. 
We will generally stick with 1D arrays to reduce confusion. 
However, do study the code below to try to understand its operation.

In [ ]:
def trajectories(init_speed,init_slopes):
    gravity = 9.8067
    max_time_aloft = 2 * init_speed / gravity 
    time = np.linspace(0,max_time_aloft,1000)
    init_speed_x = init_speed / np.sqrt(1 + init_slopes**2) 
    init_speed_y = init_speed_x * init_slopes  
    x = np.outer(init_speed_x, time)  
    y = -0.5 * gravity * time * time + np.outer(init_speed_y, time)
    # adjust x and y to stop changing once arrow hits ground
    time_of_landing = 2 * init_speed_y / gravity
    final_x = init_speed_x * time_of_landing
    final_x_2D = np.outer(final_x, np.ones(len(time)))
    x = np.minimum(x, final_x_2D)
    y = np.maximum(y,0)                        
    return time,x,y

init_speed = 70
init_slopes = np.array([1/8,1/4,1/3,1/2,2/3,3/4,1,4/3,3/2,2,3,4,8])
t,x,y = trajectories(init_speed,init_slopes)
print('shapes of t,x,y are',t.shape,x.shape,y.shape)
plt.title("Trajectories")
for i in range(len(x)):
    plt.plot(x[i],y[i])
plt.show()

Let's compute velocity on impact and time of impact for falling objects, ignoring friction.

In [ ]:
radiusEarth = 6371          # kilometers
gravityEarth = 9.807/1000   # kilometers per second squared
def velocity_on_impact(height_in_km, dt=1):    
    distance = radiusEarth + height_in_km
    speed = time = 0
    while distance > radiusEarth:
        time += dt
        gravity = gravityEarth * (radiusEarth / distance)**2
        speed +=  gravity * dt
        distance -= speed * dt
    return time,speed 

height_in_km = 1,10,100,1000,10000
for height in height_in_km:
    time,speed = velocity_on_impact(height)
    print('height',height,'km with time of impact',time,'seconds') 
    print('speed at impact =',round(speed,2),'km/sec or',round(speed*3600),'km/hour\n')

This version improves accuracy by looking at average speed in each interval. 
It also allows more flexible inputs, 
adjusts *dt* to speed computation, and displays results more neatly.

In [ ]:
radiusEarth = 6371          # kilometers
gravityEarth = 9.807/1000   # kilometers per second squared
def velocity_on_impact_better(height_in_km, radiusPlanet, gravityPlanet, dt=1):  
    distance = radiusPlanet + height_in_km
    speed = time = 0
    while distance > radiusPlanet:
        time += dt
        gravity = gravityPlanet * (radiusPlanet / distance)**2
        init_speed = speed
        speed += gravity * dt
        distance -= 0.5 * (init_speed + speed) * dt
    return time,speed
    
height = 1
while height < 1E6:
    height *= 10
    time,speed = velocity_on_impact_better(height,radiusEarth,gravityEarth)
    if time < 60:
        print('height',round(height),'km with time of impact',round(time,1),'seconds')
    elif time < 60 * 60:
        print('height',round(height),'km with time of impact',round(time/60,1),'minutes')
    elif time < 24 * 60 * 60:
        print('height',round(height),'km with time of impact',round(time/60/60,1),'hours')
    else:
        print('height',round(height),'km with time of impact',round(time/60/60/24,1),'days')
    print('speed at impact =',round(speed,2),'km/sec or',round(speed*3600),'km/hour\n')